# Validation 02 — WAV Loading & Normalization
Verifies signal is float64, mono, normalized to [-1,1].

In [1]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt, time, warnings
from pathlib import Path
from scipy.signal import butter,cheby1,cheby2,ellip,bessel,sosfiltfilt,sosfreqz,welch
from scipy.signal import spectrogram as sp_spectrogram
from scipy.io import wavfile
from itertools import product
warnings.filterwarnings('ignore')

BASE_DIR    = Path(r"D:\\1 placement\\IAESTE INTERNSHIP CZECH\\iaeste26-blasting-sound-main\\iaeste26-blasting-sound-main")
DATA_DIR    = BASE_DIR / "data"
RESULTS_DIR = BASE_DIR / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

SENSOR_PRIORITY   = ['AccAxial4507','AccRadial4507','Mic147EB','Mic46BE']
ENERGY_WINDOW_S   = 0.05;  NOISE_DURATION_S = 0.5;  ONSET_THRESHOLD = 10.0;  ONSET_OFFSET_S = 7.0
WINDOW_DURATION_S = 5.0;   WINDOW_STEP_S    = 1.0
LOWER_LIMITS = [176,225,283,353,440,565,707,880,1130,1414,1760,10,10,500,1000]
UPPER_LIMITS = [225,283,353,440,565,707,880,1130,1414,1760,2220,1000,2000,1500,2000]
N_BANDS      = len(LOWER_LIMITS)
BAND_LABELS  = [f"{lo}–{hi} Hz" for lo,hi in zip(LOWER_LIMITS,UPPER_LIMITS)]
FILTER_TYPES  = ['Butterworth','Chebyshev I','Chebyshev II','Elliptical','Bessel']
FILTER_ORDERS = [3,5,7]
CHEBY1_RIPPLE_DB=0.5; CHEBY2_ATTEN_DB=40.0; ELLIP_RIPPLE_DB=0.5; ELLIP_ATTEN_DB=40.0

P=[0]; F=[0]
def check(label, ok, note=""):
    s="[PASS]" if ok else "[FAIL]"
    if ok: P[0]+=1
    else:  F[0]+=1
    print(f"  {s}  {label}" + (f"  → {note}" if note else ""))
def info(label, val): print(f"  [INFO]  {label}: {val}")
def summary():
    t=P[0]+F[0]
    print(f"\n{'='*50}")
    print(f"  PASS: {P[0]}/{t}  |  FAIL: {F[0]}/{t}")
    print(f"  Score: {P[0]/t*100:.0f}%" if t else "  No checks run")
    print('='*50)
print("Config loaded.")


Config loaded.


In [2]:
def load_wav(path):
    fs,data=wavfile.read(path)
    if data.ndim>1: data=data[:,0]
    if   data.dtype==np.int16:  sig=data.astype(np.float64)/32768.0
    elif data.dtype==np.int32:  sig=data.astype(np.float64)/2147483648.0
    else:                       sig=data.astype(np.float64)
    return fs,sig,len(sig)/fs

def detect_onset(signal,fs):
    hop=int(ENERGY_WINDOW_S*fs); n=len(signal)//hop
    rms=np.array([np.sqrt(np.mean(signal[i*hop:(i+1)*hop]**2)) for i in range(n)])
    nf=np.mean(rms[:max(1,int(NOISE_DURATION_S/ENERGY_WINDOW_S))])
    ab=np.where(rms>ONSET_THRESHOLD*nf)[0]
    of=int(ab[0]) if len(ab) else int(np.argmax(rms))
    return of*hop, of*ENERGY_WINDOW_S, np.arange(n)*ENERGY_WINDOW_S, rms, nf

def make_windows(signal,fs):
    wl=int(WINDOW_DURATION_S*fs); sl=int(WINDOW_STEP_S*fs)
    n=max(0,(len(signal)-wl)//sl+1)
    return [signal[i*sl:i*sl+wl] for i in range(n)], np.arange(n)*WINDOW_STEP_S

def design_filter(ftype,order,lo,hi,fs):
    nyq=fs/2.; Wn=[lo/nyq,hi/nyq]
    if ftype=='Butterworth':  return butter(order,Wn,btype='bandpass',output='sos')
    if ftype=='Chebyshev I':  return cheby1(order,CHEBY1_RIPPLE_DB,Wn,btype='bandpass',output='sos')
    if ftype=='Chebyshev II': return cheby2(order,CHEBY2_ATTEN_DB,Wn,btype='bandpass',output='sos')
    if ftype=='Elliptical':   return ellip(order,ELLIP_RIPPLE_DB,ELLIP_ATTEN_DB,Wn,btype='bandpass',output='sos')
    if ftype=='Bessel':       return bessel(order,Wn,btype='bandpass',output='sos',norm='phase')

all_wavs = sorted(set(DATA_DIR.rglob("*.wav"))|set(DATA_DIR.rglob("*.WAV")))
def skey(p):
    s=Path(p).stem.split('_')[-1]
    return SENSOR_PRIORITY.index(s) if s in SENSOR_PRIORITY else 99
all_wavs = sorted(all_wavs,key=skey)
EXAMPLE_WAV = all_wavs[0] if all_wavs else None
print(f"WAV files found: {len(all_wavs)}")
if EXAMPLE_WAV: print(f"Using: {EXAMPLE_WAV.name}")


WAV files found: 1568
Using: G80_8_3_0_AccAxial4507.wav


In [3]:
if not EXAMPLE_WAV:
    print("[SKIP] No WAV files found"); raise SystemExit

fs,signal,duration = load_wav(EXAMPLE_WAV)
t = np.arange(len(signal))/fs

check("Signal dtype is float64",      signal.dtype==np.float64,      str(signal.dtype))
check("Signal is 1-D (mono)",         signal.ndim==1,                f"ndim={signal.ndim}")
check("Amplitude min ≥ -1.0",         signal.min()>=-1.0,            f"min={signal.min():.5f}")
check("Amplitude max ≤  1.0",         signal.max()<= 1.0,            f"max={signal.max():.5f}")
check("Duration > 10 s",              duration>10.0,                 f"{duration:.2f} s")
check("Sample rate ≥ 44100 Hz",       fs>=44100,                     f"{fs:,} Hz")
check("Nyquist ≥ 2000 Hz",            fs/2>=2000,                    f"{fs//2:,} Hz")
check("Time vector length = samples", len(t)==len(signal))
check("Time vector starts at 0",      t[0]==0.0)
check("No NaN in signal",             not np.any(np.isnan(signal)))
check("No Inf in signal",             not np.any(np.isinf(signal)))
info("File",         EXAMPLE_WAV.name)
info("Sample rate",  f"{fs:,} Hz")
info("Duration",     f"{duration:.2f} s")
info("Samples",      f"{len(signal):,}")
summary()


  [PASS]  Signal dtype is float64  → float64
  [PASS]  Signal is 1-D (mono)  → ndim=1
  [PASS]  Amplitude min ≥ -1.0  → min=-0.22711
  [PASS]  Amplitude max ≤  1.0  → max=0.20410
  [PASS]  Duration > 10 s  → 89.25 s
  [PASS]  Sample rate ≥ 44100 Hz  → 48,000 Hz
  [PASS]  Nyquist ≥ 2000 Hz  → 24,000 Hz
  [PASS]  Time vector length = samples
  [PASS]  Time vector starts at 0
  [PASS]  No NaN in signal
  [PASS]  No Inf in signal
  [INFO]  File: G80_8_3_0_AccAxial4507.wav
  [INFO]  Sample rate: 48,000 Hz
  [INFO]  Duration: 89.25 s
  [INFO]  Samples: 4,283,793

  PASS: 11/11  |  FAIL: 0/11
  Score: 100%
